In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import re
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
session = requests.Session()
session.headers.update({"User-Agent": "Chrome/147.0.7727.56"})

In [3]:
def getAndParseURL(url):
    for i in range(3): #Retrying 3 times if cant get any response from server
        try:
            time.sleep(random.uniform(1.5, 3.5))  #Random delay
            
            result = session.get(url, timeout=10) #If no respose leave it.
            soup = BeautifulSoup(result.text, "html.parser")
            if len(soup.find_all("tr", id=re.compile("item"))) > 0:
                return soup
            
        except:
            pass
    return None

In [4]:
airport_list = ["LHR","IST","CDG","AMS","MAD","FRA","BCN","FCO","MUC","DUB","LIS","ATH","ZRH"]
routes = []
for origin in airport_list:
    for destination in airport_list:
        if origin != destination:
            routes.append((origin,destination))
len(routes)

156

In [5]:
pages = []
for origin, destination in routes:
    for day in range(1,31):
        date_str = f"{day:02}.06.2026"
        pages.append(f"https://www.ucuzabilet.com/dis-hat-arama-sonuc?from={origin}&to={destination}&ddate={date_str}&cabintype=2&adult=1&directflightsonly=on&flightType=2")
        
len(pages)

4680

In [6]:
deneme = []
for day in range(1,31):
    date_str = f"{day:02}.06.2026"
    deneme.append(f"https://www.ucuzabilet.com/dis-hat-arama-sonuc?from=IST&to=CDG&ddate={date_str}&cabintype=2&adult=1&directflightsonly=on&flightType=2")
        
len(deneme)

30

In [7]:
total_flights = len(html.find_all("tr", id=re.compile("item"))) #Linkteki toplam uçuş kartı sayısı 
total_flights

NameError: name 'html' is not defined

In [8]:
def process_page(link):
    html = getAndParseURL(link)
    if html is None:
        return []
    
    page_data = []
    total_flights = len(html.find_all("tr", id=re.compile("item"))) #Total flights in one link
    for flight in range(1,total_flights+1):
        try:
            date = html.find("div", class_="datepicker-box-blank").input.get("value").strip()
        except:
            date = np.nan
        try:
            airline = html.find("tr", id=f"item-{flight}").find("div", class_="airline").text.strip()
        except:
            airline = np.nan
        try:
            class_type = html.find("tr", id=f"item-{flight}").find_all("div")[2].text.strip()
        except:
            class_type = np.nan
        try:
            departure_time = html.find("tr", id=f"item-{flight}").find_all("b", class_="flight-time")[0].text.strip()
        except:
            departure_time = np.nan
        try:
            arrival_time = html.find("tr", id=f"item-{flight}").find_all("b", class_="flight-time")[1].text.strip()
        except:
            arrival_time = np.nan
        try:
            departure_airport = html.find("tr", id=f"item-{flight}").find_all("div", class_="airport")[0].text.strip()
        except:
            departure_airport = np.nan
        try:
            arrival_airport = html.find("tr", id=f"item-{flight}").find_all("div", class_="airport")[1].text.strip()
        except:
            arrival_airport = np.nan
        try:
            flight_duration = html.find("tr", id=f"item-{flight}").find("span", class_="flight-duration").text.strip()
        except:
            flight_duration = np.nan
        try:
            baggage_amount = html.find("tr", id=f"flight-detail-bar-{flight}").find("div", class_="col-10 text-left pl-3").text.strip().splitlines()[0] 
        except:
            baggage_amount = np.nan
        try:
            price = html.find("tr", id=f"item-{flight}").find("i", class_="integers").text.strip()
        except:
            price = np.nan
        row = [date, airline, class_type, departure_time, arrival_time, departure_airport, arrival_airport, flight_duration, baggage_amount, price]
        page_data.append(row)
    
    return page_data


In [9]:
data = []

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_page, link) for link in pages]
    
    for future in as_completed(futures):
        result = future.result()
        if result:
            data.extend(result)

In [10]:
columns = ["date", "airline", "class_type", "departure_time", "arrival_time", "departure_airport", "arrival_airport", "flight_duration", "baggage_amount", "price"]
    
df = pd.DataFrame(data, columns=columns)
df

,date,airline,class_type,departure_time,arrival_time,departure_airport,arrival_airport,flight_duration,baggage_amount,price
0,07.06.2026,British Airways,Ekonomi,19:05,00:55*,LHR,IST,3sa 50dk,Sadece kabin bagajı,8128
1,07.06.2026,Türk Hava Yolları,Ekonomi,22:40,04:20*,LHR,IST,3sa 40dk,Sadece kabin bagajı,8642
2,07.06.2026,Türk Hava Yolları,Ekonomi,18:30,00:15*,LHR,IST,3sa 45dk,Sadece kabin bagajı,8642
3,07.06.2026,Türk Hava Yolları,Ekonomi,13:20,19:10,LHR,IST,3sa 50dk,Sadece kabin bagajı,8642
4,07.06.2026,Türk Hava Yolları,Ekonomi,16:45,22:35,LHR,IST,3sa 50dk,Sadece kabin bagajı,8642
...,...,...,...,...,...,...,...,...,...,...
42918,30.06.2026,Swiss Int. Airlines,Business,21:20,00:55*,ZRH,ATH,2sa 35dk,2 x 23 kg,24661
42919,30.06.2026,Swiss Int. Airlines,Business,09:25,13:10,ZRH,ATH,2sa 45dk,2 x 23 kg,29613
42920,30.06.2026,Aegean Airlines,Business,12:05,15:45,ZRH,ATH,2sa 40dk,2 x 23 kg,29879
42921,30.06.2026,Aegean Airlines,Business,21:20,00:55*,ZRH,ATH,2sa 35dk,2 x 23 kg,29879


In [10]:
df = pd.read_csv("flight_data.csv", index_col=0)
df

,date,airline,class_type,departure_time,arrival_time,departure_airport,arrival_airport,flight_duration,baggage_amount,price
0,07.06.2026,British Airways,Ekonomi,19:05,00:55*,LHR,IST,3sa 50dk,Sadece kabin bagajı,8128
1,07.06.2026,Türk Hava Yolları,Ekonomi,22:40,04:20*,LHR,IST,3sa 40dk,Sadece kabin bagajı,8642
2,07.06.2026,Türk Hava Yolları,Ekonomi,18:30,00:15*,LHR,IST,3sa 45dk,Sadece kabin bagajı,8642
3,07.06.2026,Türk Hava Yolları,Ekonomi,13:20,19:10,LHR,IST,3sa 50dk,Sadece kabin bagajı,8642
4,07.06.2026,Türk Hava Yolları,Ekonomi,16:45,22:35,LHR,IST,3sa 50dk,Sadece kabin bagajı,8642
...,...,...,...,...,...,...,...,...,...,...
42918,30.06.2026,Swiss Int. Airlines,Business,21:20,00:55*,ZRH,ATH,2sa 35dk,2 x 23 kg,24661
42919,30.06.2026,Swiss Int. Airlines,Business,09:25,13:10,ZRH,ATH,2sa 45dk,2 x 23 kg,29613
42920,30.06.2026,Aegean Airlines,Business,12:05,15:45,ZRH,ATH,2sa 40dk,2 x 23 kg,29879
42921,30.06.2026,Aegean Airlines,Business,21:20,00:55*,ZRH,ATH,2sa 35dk,2 x 23 kg,29879


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42923 entries, 0 to 42922
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   date               42923 non-null  object
 1   airline            42923 non-null  object
 2   class_type         42923 non-null  object
 3   departure_time     42923 non-null  object
 4   arrival_time       42923 non-null  object
 5   departure_airport  42923 non-null  object
 6   arrival_airport    42923 non-null  object
 7   flight_duration    42923 non-null  object
 8   baggage_amount     42923 non-null  object
 9   price              42923 non-null  int64 
dtypes: int64(1), object(9)
memory usage: 3.6+ MB


In [12]:
df.to_csv("flight_data.csv")

In [13]:
uncompleted = df.groupby(["departure_airport","arrival_airport","date"])["price"].count().to_frame().reset_index()

In [14]:
uncompleted

,departure_airport,arrival_airport,date,price
0,AMS,ATH,03.06.2026,10
1,AMS,ATH,04.06.2026,11
2,AMS,ATH,07.06.2026,11
3,AMS,ATH,09.06.2026,11
4,AMS,ATH,11.06.2026,11
...,...,...,...,...
2243,ZRH,MUC,23.06.2026,24
2244,ZRH,MUC,24.06.2026,24
2245,ZRH,MUC,25.06.2026,24
2246,ZRH,MUC,26.06.2026,24


In [15]:
dates = pd.date_range("2026-06-01", "2026-06-30").strftime("%d.%m.%Y")
dates

Index(['01.06.2026', '02.06.2026', '03.06.2026', '04.06.2026', '05.06.2026',
       '06.06.2026', '07.06.2026', '08.06.2026', '09.06.2026', '10.06.2026',
       '11.06.2026', '12.06.2026', '13.06.2026', '14.06.2026', '15.06.2026',
       '16.06.2026', '17.06.2026', '18.06.2026', '19.06.2026', '20.06.2026',
       '21.06.2026', '22.06.2026', '23.06.2026', '24.06.2026', '25.06.2026',
       '26.06.2026', '27.06.2026', '28.06.2026', '29.06.2026', '30.06.2026'],
      dtype='object')

In [16]:
route = pd.DataFrame(routes, columns=["departure_airport","arrival_airport"])

In [17]:
full_grid = route.merge(pd.DataFrame({"date": dates}), how="cross")
full_grid

,departure_airport,arrival_airport,date
0,LHR,IST,01.06.2026
1,LHR,IST,02.06.2026
2,LHR,IST,03.06.2026
3,LHR,IST,04.06.2026
4,LHR,IST,05.06.2026
...,...,...,...
4675,ZRH,ATH,26.06.2026
4676,ZRH,ATH,27.06.2026
4677,ZRH,ATH,28.06.2026
4678,ZRH,ATH,29.06.2026


In [18]:
agg = (df.groupby(["departure_airport","arrival_airport","date"]).size().reset_index(name="count"))

merged = full_grid.merge(agg, how="left")

In [19]:
merged

,departure_airport,arrival_airport,date,count
0,LHR,IST,01.06.2026,NaN
1,LHR,IST,02.06.2026,NaN
2,LHR,IST,03.06.2026,NaN
3,LHR,IST,04.06.2026,NaN
4,LHR,IST,05.06.2026,NaN
...,...,...,...,...
4675,ZRH,ATH,26.06.2026,NaN
4676,ZRH,ATH,27.06.2026,NaN
4677,ZRH,ATH,28.06.2026,NaN
4678,ZRH,ATH,29.06.2026,20.0


In [20]:
empty = merged[merged["count"].isnull()]
empty.date = pd.to_datetime(empty.date, format="%d.%m.%Y")

C:\Users\Ozi\AppData\Local\Temp\ipykernel_39132\1339523011.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  empty.date = pd.to_datetime(empty.date, format="%d.%m.%Y")


In [21]:
empty["day"] = empty.date.dt.day

C:\Users\Ozi\AppData\Local\Temp\ipykernel_39132\3863179389.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  empty["day"] = empty.date.dt.day


In [22]:
empty.day = empty.day.astype(str)

C:\Users\Ozi\AppData\Local\Temp\ipykernel_39132\1205362050.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  empty.day = empty.day.astype(str)


In [23]:
empty.loc[empty["day"]=="1", "day"] = "01"
empty.loc[empty["day"]=="2", "day"] = "02"
empty.loc[empty["day"]=="3", "day"] = "03"
empty.loc[empty["day"]=="4", "day"] = "04"
empty.loc[empty["day"]=="5", "day"] = "05"
empty.loc[empty["day"]=="6", "day"] = "06"
empty.loc[empty["day"]=="7", "day"] = "07"
empty.loc[empty["day"]=="8", "day"] = "08"
empty.loc[empty["day"]=="9", "day"] = "09"

In [24]:
empty.day

0       01
1       02
2       03
3       04
4       05
        ..
4672    23
4674    25
4675    26
4676    27
4677    28
Name: day, Length: 2432, dtype: object

In [25]:
empty

,departure_airport,arrival_airport,date,count,day
0,LHR,IST,2026-06-01,NaN,01
1,LHR,IST,2026-06-02,NaN,02
2,LHR,IST,2026-06-03,NaN,03
3,LHR,IST,2026-06-04,NaN,04
4,LHR,IST,2026-06-05,NaN,05
...,...,...,...,...,...
4672,ZRH,ATH,2026-06-23,NaN,23
4674,ZRH,ATH,2026-06-25,NaN,25
4675,ZRH,ATH,2026-06-26,NaN,26
4676,ZRH,ATH,2026-06-27,NaN,27


In [26]:
empty.iloc[0,[0,1,-1]].values

array(['LHR', 'IST', '01'], dtype=object)

In [27]:
missed_list = []
for row in range(0, len(empty)):
    row_data = empty.iloc[row,[0,1,-1]].values
    missed_list.append(row_data)

missed_list

[array(['LHR', 'IST', '01'], dtype=object),
 array(['LHR', 'IST', '02'], dtype=object),
 array(['LHR', 'IST', '03'], dtype=object),
 array(['LHR', 'IST', '04'], dtype=object),
 array(['LHR', 'IST', '05'], dtype=object),
 array(['LHR', 'IST', '06'], dtype=object),
 array(['LHR', 'IST', '08'], dtype=object),
 array(['LHR', 'IST', '09'], dtype=object),
 array(['LHR', 'IST', '11'], dtype=object),
 array(['LHR', 'IST', '12'], dtype=object),
 array(['LHR', 'IST', '15'], dtype=object),
 array(['LHR', 'IST', '16'], dtype=object),
 array(['LHR', 'IST', '19'], dtype=object),
 array(['LHR', 'IST', '20'], dtype=object),
 array(['LHR', 'IST', '23'], dtype=object),
 array(['LHR', 'IST', '24'], dtype=object),
 array(['LHR', 'IST', '26'], dtype=object),
 array(['LHR', 'IST', '28'], dtype=object),
 array(['LHR', 'IST', '29'], dtype=object),
 array(['LHR', 'CDG', '01'], dtype=object),
 array(['LHR', 'CDG', '02'], dtype=object),
 array(['LHR', 'CDG', '03'], dtype=object),
 array(['LHR', 'CDG', '04'], dty

In [28]:
missed_pages = []
for origin, destination, day in missed_list:
    missed_pages.append(f"https://www.ucuzabilet.com/dis-hat-arama-sonuc?from={origin}&to={destination}&ddate={day}.06.2026&cabintype=2&adult=1&directflightsonly=on&flightType=2")
        
len(missed_pages)

2432

In [29]:
missed_data = []

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_page, link) for link in missed_pages]
    
    for future in as_completed(futures):
        result = future.result()
        if result:
            missed_data.extend(result)

In [30]:
columns = ["date", "airline", "class_type", "departure_time", "arrival_time", "departure_airport", "arrival_airport", "flight_duration", "baggage_amount", "price"]
    
missed_df = pd.DataFrame(missed_data, columns=columns)
missed_df

,date,airline,class_type,departure_time,arrival_time,departure_airport,arrival_airport,flight_duration,baggage_amount,price
0,12.06.2026,Türk Hava Yolları,Ekonomi,22:40,04:20*,LHR,IST,3sa 40dk,Sadece kabin bagajı,8021
1,12.06.2026,Türk Hava Yolları,Ekonomi,18:30,00:15*,LHR,IST,3sa 45dk,Sadece kabin bagajı,8021
2,12.06.2026,Türk Hava Yolları,Ekonomi,16:45,22:35,LHR,IST,3sa 50dk,Sadece kabin bagajı,8021
3,12.06.2026,Türk Hava Yolları,Ekonomi,06:30,12:25,LHR,IST,3sa 55dk,Sadece kabin bagajı,8021
4,12.06.2026,British Airways,Ekonomi,18:50,00:40*,LHR,IST,3sa 50dk,Sadece kabin bagajı,8128
...,...,...,...,...,...,...,...,...,...,...
41614,27.06.2026,Swiss Int. Airlines,Business,12:05,15:45,ZRH,ATH,2sa 40dk,2 x 23 kg,37775
41615,27.06.2026,Aegean Airlines,Business,16:35,20:10,ZRH,ATH,2sa 35dk,2 x 23 kg,38040
41616,27.06.2026,Aegean Airlines,Business,11:15,14:50,ZRH,ATH,2sa 35dk,2 x 23 kg,44076
41617,27.06.2026,Swiss Int. Airlines,Business,11:15,14:50,ZRH,ATH,2sa 35dk,2 x 23 kg,51095


In [31]:
df_final = pd.concat([df, missed_df], axis=0, ignore_index=True)

In [32]:
df_final

,date,airline,class_type,departure_time,arrival_time,departure_airport,arrival_airport,flight_duration,baggage_amount,price
0,07.06.2026,British Airways,Ekonomi,19:05,00:55*,LHR,IST,3sa 50dk,Sadece kabin bagajı,8128
1,07.06.2026,Türk Hava Yolları,Ekonomi,22:40,04:20*,LHR,IST,3sa 40dk,Sadece kabin bagajı,8642
2,07.06.2026,Türk Hava Yolları,Ekonomi,18:30,00:15*,LHR,IST,3sa 45dk,Sadece kabin bagajı,8642
3,07.06.2026,Türk Hava Yolları,Ekonomi,13:20,19:10,LHR,IST,3sa 50dk,Sadece kabin bagajı,8642
4,07.06.2026,Türk Hava Yolları,Ekonomi,16:45,22:35,LHR,IST,3sa 50dk,Sadece kabin bagajı,8642
...,...,...,...,...,...,...,...,...,...,...
84537,27.06.2026,Swiss Int. Airlines,Business,12:05,15:45,ZRH,ATH,2sa 40dk,2 x 23 kg,37775
84538,27.06.2026,Aegean Airlines,Business,16:35,20:10,ZRH,ATH,2sa 35dk,2 x 23 kg,38040
84539,27.06.2026,Aegean Airlines,Business,11:15,14:50,ZRH,ATH,2sa 35dk,2 x 23 kg,44076
84540,27.06.2026,Swiss Int. Airlines,Business,11:15,14:50,ZRH,ATH,2sa 35dk,2 x 23 kg,51095


In [33]:
df_final.to_csv("flight_data.csv")

In [34]:
agg = (df_final.groupby(["departure_airport","arrival_airport","date"]).size().reset_index(name="count"))

merged = full_grid.merge(agg, how="left")

In [36]:
merged[merged["count"].isnull()]

,departure_airport,arrival_airport,date,count
0,LHR,IST,01.06.2026,NaN
1,LHR,IST,02.06.2026,NaN
2,LHR,IST,03.06.2026,NaN
3,LHR,IST,04.06.2026,NaN
4,LHR,IST,05.06.2026,NaN
...,...,...,...,...
4627,ZRH,LIS,08.06.2026,NaN
4643,ZRH,LIS,24.06.2026,NaN
4659,ZRH,ATH,10.06.2026,NaN
4669,ZRH,ATH,20.06.2026,NaN


In [37]:
empty = merged[merged["count"].isnull()]
empty.date = pd.to_datetime(empty.date, format="%d.%m.%Y")
empty["day"] = empty.date.dt.day
empty.day = empty.day.astype(str)
empty.loc[empty["day"]=="1", "day"] = "01"
empty.loc[empty["day"]=="2", "day"] = "02"
empty.loc[empty["day"]=="3", "day"] = "03"
empty.loc[empty["day"]=="4", "day"] = "04"
empty.loc[empty["day"]=="5", "day"] = "05"
empty.loc[empty["day"]=="6", "day"] = "06"
empty.loc[empty["day"]=="7", "day"] = "07"
empty.loc[empty["day"]=="8", "day"] = "08"
empty.loc[empty["day"]=="9", "day"] = "09"

C:\Users\Ozi\AppData\Local\Temp\ipykernel_39132\1665432390.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  empty.date = pd.to_datetime(empty.date, format="%d.%m.%Y")
C:\Users\Ozi\AppData\Local\Temp\ipykernel_39132\1665432390.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  empty["day"] = empty.date.dt.day
C:\Users\Ozi\AppData\Local\Temp\ipykernel_39132\1665432390.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] =

In [38]:
empty

,departure_airport,arrival_airport,date,count,day
0,LHR,IST,2026-06-01,NaN,01
1,LHR,IST,2026-06-02,NaN,02
2,LHR,IST,2026-06-03,NaN,03
3,LHR,IST,2026-06-04,NaN,04
4,LHR,IST,2026-06-05,NaN,05
...,...,...,...,...,...
4627,ZRH,LIS,2026-06-08,NaN,08
4643,ZRH,LIS,2026-06-24,NaN,24
4659,ZRH,ATH,2026-06-10,NaN,10
4669,ZRH,ATH,2026-06-20,NaN,20


In [39]:
missed_list = []
for row in range(0, len(empty)):
    row_data = empty.iloc[row,[0,1,-1]].values
    missed_list.append(row_data)

missed_list

[array(['LHR', 'IST', '01'], dtype=object),
 array(['LHR', 'IST', '02'], dtype=object),
 array(['LHR', 'IST', '03'], dtype=object),
 array(['LHR', 'IST', '04'], dtype=object),
 array(['LHR', 'IST', '05'], dtype=object),
 array(['AMS', 'ATH', '26'], dtype=object),
 array(['AMS', 'ZRH', '04'], dtype=object),
 array(['MAD', 'CDG', '02'], dtype=object),
 array(['MAD', 'CDG', '24'], dtype=object),
 array(['MAD', 'CDG', '26'], dtype=object),
 array(['MAD', 'CDG', '30'], dtype=object),
 array(['MAD', 'AMS', '18'], dtype=object),
 array(['MAD', 'FRA', '06'], dtype=object),
 array(['MAD', 'FRA', '18'], dtype=object),
 array(['MAD', 'BCN', '05'], dtype=object),
 array(['MAD', 'BCN', '24'], dtype=object),
 array(['MAD', 'FCO', '07'], dtype=object),
 array(['MAD', 'MUC', '12'], dtype=object),
 array(['MAD', 'DUB', '13'], dtype=object),
 array(['MAD', 'DUB', '20'], dtype=object),
 array(['MAD', 'LIS', '06'], dtype=object),
 array(['MAD', 'ATH', '11'], dtype=object),
 array(['MAD', 'ATH', '19'], dty

In [40]:
missed_pages = []
for origin, destination, day in missed_list:
    missed_pages.append(f"https://www.ucuzabilet.com/dis-hat-arama-sonuc?from={origin}&to={destination}&ddate={day}.06.2026&cabintype=2&adult=1&directflightsonly=on&flightType=2")
        
len(missed_pages)

193

In [41]:
missed_data = []

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_page, link) for link in missed_pages]
    
    for future in as_completed(futures):
        result = future.result()
        if result:
            missed_data.extend(result)

In [42]:
columns = ["date", "airline", "class_type", "departure_time", "arrival_time", "departure_airport", "arrival_airport", "flight_duration", "baggage_amount", "price"]
    
missed_df_2 = pd.DataFrame(missed_data, columns=columns)
missed_df_2

,date,airline,class_type,departure_time,arrival_time,departure_airport,arrival_airport,flight_duration,baggage_amount,price
0,01.06.2026,British Airways,Ekonomi,18:35,00:25*,LHR,IST,3sa 50dk,Sadece kabin bagajı,8128
1,01.06.2026,Türk Hava Yolları,Ekonomi,22:40,04:20*,LHR,IST,3sa 40dk,Sadece kabin bagajı,9885
2,01.06.2026,Türk Hava Yolları,Ekonomi,18:30,00:15*,LHR,IST,3sa 45dk,Sadece kabin bagajı,9885
3,01.06.2026,Türk Hava Yolları,Ekonomi,16:45,22:35,LHR,IST,3sa 50dk,Sadece kabin bagajı,9885
4,01.06.2026,Türk Hava Yolları,Ekonomi,06:30,12:25,LHR,IST,3sa 55dk,Sadece kabin bagajı,9885
...,...,...,...,...,...,...,...,...,...,...
2748,20.06.2026,Swiss Int. Airlines,Business,12:05,15:45,ZRH,ATH,2sa 40dk,2 x 23 kg,37775
2749,20.06.2026,Swiss Int. Airlines,Business,16:35,20:10,ZRH,ATH,2sa 35dk,2 x 23 kg,37775
2750,20.06.2026,Aegean Airlines,Business,11:15,14:50,ZRH,ATH,2sa 35dk,2 x 23 kg,44076
2751,20.06.2026,Swiss Int. Airlines,Business,11:15,14:50,ZRH,ATH,2sa 35dk,2 x 23 kg,51095


In [43]:
df_final = pd.concat([df_final, missed_df_2], axis=0, ignore_index=True)

In [45]:
df_final.to_csv("flight_data.csv")

In [46]:
agg = (df_final.groupby(["departure_airport","arrival_airport","date"]).size().reset_index(name="count"))

merged = full_grid.merge(agg, how="left")

In [47]:
merged[merged["count"].isnull()]

,departure_airport,arrival_airport,date,count
